In [0]:
# Cell 1 — ALL imports for the entire notebook
from pyspark.sql.functions import (
    col, sum, count, avg, round, rank,
    countDistinct, when, to_date,
    current_timestamp, lit
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
'''Total Revenue                   → SUM(order_value)
Avg Revenue Per Order           → AVG(order_value)
Avg Revenue Per Item            → SUM(order_value) / SUM(item_count)
Top Performing Restaurant       → RANK() by revenue
Revenue Trends Over Time        → daily/weekly rolling SUM
Revenue By Customer Segment     → GROUP BY rfm_segment
Revenue By Restaurant & Location → GROUP BY city, cuisine
Delivery Performance            → avg actual_time, SLA breach %
Geographic Revenue Insight      → revenue by city + zone'''

SILVER_ORDERS = "swiggy.silver.orders_clean"
SILVER_DELIVERY = "swiggy.silver.delivery_clean"
SILVER_RESTAURANTS = "swiggy.silver.restaurants"
SILVER_CUSTOMERS = "swiggy.silver.customers"

GOLD_DAILY_REVENUE = "swiggy.gold.daily_revenue"
GOLD_RESTAURANT_KPIS = "swiggy.gold.restaurant_kpi"
GOLD_DELIVERY_KPIS = "swiggy.gold.delivery_kpi"
GOLD_CUSTOMER_SEGMENTS = "swiggy.gold.customer_segments"
GOLD_CITY_DEMAND = "swiggy.gold.city_demand"


In [0]:
orders = spark.table(SILVER_ORDERS).filter(col("status") == "delivered")

daily_revenue = (
    orders
    .withColumn("order_date" , to_date(col("placed_at")))
    .groupBy("order_date", "city")
    .agg(round(sum("order_value"),2).alias("total_revenue"),
         count("order_id").alias("total_orders"),
         round(avg("order_value"),2).alias("avg_revenue_per_order"),
         round(sum("order_value") / sum("item_count"),2).alias("avg_revenue_per_item"),
         round(sum("discount_amount"),2).alias("total_discount"))
         )
         


daily_revenue.write \
             .format("delta")\
             .mode("overwrite")\
             .saveAsTable(GOLD_DAILY_REVENUE)
print(f"daily revenue rows:{spark.table(GOLD_DAILY_REVENUE).count()}")


In [0]:
orders = spark.table(SILVER_ORDERS)
restaurant = spark.table(SILVER_RESTAURANTS).filter(col("is_current") == True)
joined = orders.join(restaurant, "restaurant_id" , "left")
restaurant_kpi = (
    joined
    .groupBy("restaurant_id","restaurant_name","cuisine_type" , "restaurant_city")
    .agg(round(sum("order_value"),2).alias("total_revenue"),
         count("order_id").alias("total_orders"),
         round(avg("order_value"),2).alias("avg_order_value"),
         count(when(col("status")== "cancelled", True)).alias("cancelled_orders")
         ))
window_spec = Window.partitionBy("restaurant_city").orderBy(col("total_revenue").desc())
restaurant_kpi = restaurant_kpi.withColumn("rank", rank().over(window_spec))    
restaurant_kpi.write \
             .format("delta")\
             .mode("overwrite")\
             .saveAsTable(GOLD_RESTAURANT_KPIS)
print(f"restaurant kpi rows:{spark.table(GOLD_RESTAURANT_KPIS).count()}")


In [0]:
delivery = spark.table(SILVER_DELIVERY).filter(col("status") == "delivered")
delivery_kpi = (
    delivery
    .groupBy("city")
    .agg(count("order_id").alias("total_deliveries"),
         round(avg("promised_time_min"),2).alias("avg_promise_time"),
         round(avg("actual_time_min"),2).alias("avg_actual_time"),
         round(avg("distance_km"),2).alias("avg_distance_km"),
         count(when(col("is_late") == True, True)).alias("late_deliveries"))
    .withColumn("sla_breach_pct", round(col("late_deliveries") / col("total_deliveries") * 100,1)))

delivery_kpi.write\
            .format("delta")\
            .mode("overwrite")\
            .saveAsTable(GOLD_DELIVERY_KPIS)
print(f"delivery kpi rows:{spark.table(GOLD_DELIVERY_KPIS).count()}")   

In [0]:
spark.table(GOLD_DELIVERY_KPIS) \
     .orderBy(col("sla_breach_pct").desc()) \
     .show(5, truncate=False)

In [0]:
orders = spark.table(SILVER_ORDERS).filter(col("status") == "delivered")
customers = spark.table(SILVER_CUSTOMERS).filter(col("is_current") == True)
joined = orders.join(customers, "customer_id")
customer_segments = (
    joined
    .groupBy("rfm_segment")
    .agg(countDistinct("customer_id").alias("total_customers"),
         count("order_id").alias("total_orders"),
         round(sum("order_value"),2).alias("total_revenue"),
         round(avg("order_value"),2).alias("avg_order_value"),
         round(count("order_id") / countDistinct("customer_id"),2).alias("avg_orders_per_customer")))
customer_segments.write \
                  .format("delta")\
                  .mode("overwrite")\
                  .saveAsTable(GOLD_CUSTOMER_SEGMENTS)
print(f"customer segments rows:{spark.table(GOLD_CUSTOMER_SEGMENTS).count()}")
         

In [0]:
spark.table(GOLD_CUSTOMER_SEGMENTS) \
     .orderBy(col("total_revenue").desc()) \
     .show(truncate=False)

In [0]:
orders = spark.table(SILVER_ORDERS).filter(col("status") == "delivered")
restaurant = spark.table(SILVER_RESTAURANTS).filter(col("is_current") == True)
joined = orders.join(restaurant, "restaurant_id" )
city_demand = (
    joined
    .groupBy("city" , "zone")
    .agg(
         count("order_id").alias("total_orders"),
         round(sum("order_value"),2).alias("total_revenue"),
         round(avg("order_value"),2).alias("avg_order_value"),
         round(sum("discount_amount"),2).alias("total_discount"),
         countDistinct("restaurant_id").alias("unique_restaurants"),
         countDistinct("customer_id").alias("unique_customers")
         ))
city_demand.write \
             .format("delta")\
             .mode("overwrite")\
             .saveAsTable(GOLD_CITY_DEMAND)
print(f"city demand rows:{spark.table(GOLD_CITY_DEMAND).count()}")

In [0]:
print("=" * 45)
print("   SWIGGY PIPELINE — COMPLETE STATUS")
print("=" * 45)

print("\n--- Bronze ---")
print(f"orders_raw      : {spark.table('swiggy.bronze.orders_raw').count():>6} rows")
print(f"delivery_raw    : {spark.table('swiggy.bronze.delivery_raw').count():>6} rows")
print(f"restaurants     : {spark.table('swiggy.bronze.restaurants').count():>6} rows")
print(f"customers       : {spark.table('swiggy.bronze.customers').count():>6} rows")

print("\n--- Silver ---")
print(f"orders_clean    : {spark.table('swiggy.silver.orders_clean').count():>6} rows")
print(f"delivery_clean  : {spark.table('swiggy.silver.delivery_clean').count():>6} rows")
print(f"restaurants     : {spark.table('swiggy.silver.restaurants').count():>6} rows")
print(f"customers       : {spark.table('swiggy.silver.customers').count():>6} rows")

print("\n--- Gold ---")
print(f"daily_revenue   : {spark.table('swiggy.gold.daily_revenue').count():>6} rows")
print(f"restaurant_kpis : {spark.table('swiggy.gold.restaurant_kpi').count():>6} rows")
print(f"delivery_kpis   : {spark.table('swiggy.gold.delivery_kpi').count():>6} rows")
print(f"customer_segments:{spark.table('swiggy.gold.customer_segments').count():>6} rows")
print(f"city_demand     : {spark.table('swiggy.gold.city_demand').count():>6} rows")
print("=" * 45)